# Source-Free Adaptation for IoT IDS — Phase 1: is the failure robust?
Phase 0 (NF-ToN-IoT-v2 ↔ NF-UNSW-NB15-v2) showed a 0.41–0.69 macro-F1 gap and that BN-Adapt / TENT / SHOT recover almost none of it (TENT collapses to "all benign" on UNSW→ToN). Before designing a method, Phase 1 checks whether that finding survives four objections a reviewer will raise:

| Objection | Test |
|---|---|
| "Only one dataset pair" | add NF-BoT-IoT-v2 and NF-CSE-CIC-IDS2018-v2 when available on Kaggle (up to 12 directions) |
| "The gap is just testbed artifacts (TTL, TCP window)" | every run repeated with those features removed |
| "SFDA baselines were badly tuned" | learning-rate sweep for TENT and SHOT, plus the best value chosen *with target labels* (an upper bound no real deployment can reach) |
| "Stronger / simpler baselines exist" | AaD (neighbourhood-consistency SFDA, Yang et al., NeurIPS 2022) and EM label-shift correction (Saerens et al., 2002), both source-free |

Priors: the dataset's natural attack rate and a benign-dominated 5%. Results go to Drive (`sfda_ids_phase1`), resumable.

In [8]:
# ===== 1. Setup =====
import os, glob, json, time, math, random, copy
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import f1_score
try:
    from google.colab import drive
    drive.mount('/content/drive'); ROOT = '/content/drive/MyDrive'
except Exception:
    ROOT = '.'
OUT = f'{ROOT}/sfda_ids_phase1'; os.makedirs(OUT, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| results ->', OUT)

CFG = dict(
    datasets={'ton': 'dhoogla/nftoniotv2', 'unsw': 'dhoogla/nfunswnb15v2',
              'bot': 'dhoogla/nfbotiotv2', 'cic': 'dhoogla/nfcsecicids2018v2'},
    required=['ton', 'unsw'],
    known_rows={'ton': 16_940_496, 'unsw': 2_390_275, 'bot': 37_763_497, 'cic': 18_893_708},
    attack_rate={'ton': 0.6399, 'unsw': 0.0398, 'bot': 0.9964, 'cic': 0.1195},   # published NF-v2 rates (UQ)
    n_per_dataset=300_000, test_frac=0.3, seeds=[0, 1, 2],
    hidden=[128, 64], src_epochs=5, bs=512, lr=1e-3,
    priors=['natural', 0.05], n_target_train=100_000, n_target_test=40_000,
    featsets=['all', 'no_artifacts'], artifacts=['MIN_TTL', 'MAX_TTL', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT'],
    tent_lrs=[1e-4, 1e-3, 1e-2], shot_lrs=[1e-5, 1e-4, 1e-3], default_tent_lr=1e-3, default_shot_lr=1e-4,
    tent_epochs=1, shot_epochs=3, shot_beta=0.3, aad_epochs=3, aad_lr=1e-4, aad_k=5,
    harm_margin=0.05, recovery_min=0.5)
json.dump(CFG, open(f'{OUT}/config.json', 'w'), indent=2)
DROP = ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'L4_SRC_PORT', 'L4_DST_PORT', 'DNS_QUERY_ID', 'Dataset']
NEG = ['0', '0.0', 'benign', 'false', 'normal']
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
device: cuda | results -> /content/drive/MyDrive/sfda_ids_phase1


In [9]:
# ===== 2. Data: class-aware sampling so rare classes (benign in BoT-IoT) are represented =====
def label_of(df):
    cols = {c.lower(): c for c in df.columns}; lab = cols.get('label')
    assert lab is not None, f'no Label column in {list(df.columns)}'
    return (~df[lab].astype(str).str.strip().str.lower().isin(NEG)).values

def load_nf(key, slug, n, seed=0):
    cache = f'{OUT}/data_{key}_{n}.parquet'
    if os.path.exists(cache):
        print(key, 'loaded from cache'); return pd.read_parquet(cache)
    import kagglehub
    root = kagglehub.dataset_download(slug)
    files = sorted(glob.glob(os.path.join(root, '**', '*.parquet'), recursive=True)) or sorted(glob.glob(os.path.join(root, '**', '*.csv'), recursive=True))
    assert files, f'no parquet/csv under {root}'
    print(key, 'files:', [os.path.basename(f) for f in files])
    r = CFG['attack_rate'][key]; rs = np.random.RandomState(seed); parts = []
    def keep(d, tot):
        y = label_of(d); pa = min(1.0, 0.65 * n / (tot * r)); pb = min(1.0, 0.65 * n / (tot * (1 - r)))
        return d[rs.rand(len(d)) < np.where(y, pa, pb)]
    if files[0].endswith('.parquet'):
        import pyarrow.parquet as pq
        pfs = [pq.ParquetFile(f) for f in files]; tot = sum(p.metadata.num_rows for p in pfs)
        for p in pfs:
            for b in p.iter_batches(batch_size=1_000_000): parts.append(keep(b.to_pandas(), tot))
    else:
        tot = CFG['known_rows'][key]
        for f in files:
            for d in pd.read_csv(f, chunksize=1_000_000, low_memory=False): parts.append(keep(d, tot))
    df = pd.concat(parts, ignore_index=True)
    if len(df) > n: df = df.sample(n=n, random_state=seed).reset_index(drop=True)
    try: df.to_parquet(cache)
    except Exception as e: print('cache not written:', e)
    print(key, 'sampled', len(df), 'rows, attack share in sample', round(float(label_of(df).mean()), 3)); return df

def prepare(df):
    cols = {c.lower(): c for c in df.columns}; lab, att = cols.get('label'), cols.get('attack')
    y = label_of(df).astype(np.int64)
    a = df[att].astype(str).str.strip().str.lower().values if att else np.where(y == 1, 'attack', 'benign')
    X = df.drop(columns=[c for c in [lab, att] + DROP if c is not None and c in df.columns])
    X = X.apply(lambda c: pd.to_numeric(c.astype(str) if str(c.dtype) in ('category', 'object', 'bool') else c, errors='coerce'))
    return X.loc[:, X.notna().any()].fillna(0.0), y, a

def build_data(raw):
    prep = {k: prepare(v) for k, v in raw.items()}
    common = sorted(set.intersection(*[set(p[0].columns) for p in prep.values()]))
    data = {}
    for k, (X, y, a) in prep.items():
        V = X[common].values; Xl = (np.sign(V) * np.log1p(np.abs(V))).astype(np.float32)
        idx = np.random.RandomState(0).permutation(len(y)); nt = int(CFG['test_frac'] * len(y))
        te, tr = idx[:nt], idx[nt:]
        data[k] = dict(Xtr=Xl[tr], ytr=y[tr], atr=a[tr], Xte=Xl[te], yte=y[te], ate=a[te])
        print(f'{k}: {len(y)} rows | types {sorted(set(a[y == 1]))}')
    print(f'{len(common)} common features; artifacts present: {[c for c in CFG["artifacts"] if c in common]}')
    return data, common

In [10]:
# ===== 3. Model, SFDA methods, metrics =====
class Net(nn.Module):
    def __init__(s, d, h, k=2):
        super().__init__()
        s.f = nn.Sequential(nn.Linear(d, h[0]), nn.BatchNorm1d(h[0]), nn.ReLU(), nn.Linear(h[0], h[1]), nn.BatchNorm1d(h[1]), nn.ReLU())
        s.h = nn.Linear(h[1], k)
    def forward(s, x): return s.h(s.f(x))
def T(x): return torch.as_tensor(x, device=device)
def batches(n, bs, gen=None):
    perm = torch.randperm(n, generator=gen) if gen is not None else torch.randperm(n)
    perm = perm.to(device)
    return [perm[i:i + bs] for i in range(0, n, bs) if len(perm[i:i + bs]) > 1]

def train_sup(X, y, seed, epochs):
    set_seed(seed); m = Net(X.shape[1], CFG['hidden']).to(device); opt = torch.optim.Adam(m.parameters(), CFG['lr'])
    Xt, yt = T(X), T(y); g = torch.Generator().manual_seed(seed)
    for _ in range(epochs):
        m.train()
        for b in batches(len(Xt), CFG['bs'], g):
            opt.zero_grad(); F.cross_entropy(m(Xt[b]), yt[b]).backward(); opt.step()
    return m.eval()

@torch.no_grad()
def proba(m, X):
    m.eval(); Xt = T(X)
    return torch.cat([F.softmax(m(Xt[i:i + 8192]), 1) for i in range(0, len(Xt), 8192)]).cpu().numpy()

def bn_adapt(m, X):
    m = copy.deepcopy(m)
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm1d): mod.reset_running_stats(); mod.momentum = None
    m.train(); Xt = T(X)
    with torch.no_grad():
        for i in range(0, len(Xt), 1024):
            if len(Xt[i:i + 1024]) > 1: m(Xt[i:i + 1024])
    return m.eval()

def tent(m, X, seed, lr):
    set_seed(seed); m = copy.deepcopy(m); m.requires_grad_(False); params = []
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm1d): mod.requires_grad_(True); params += [mod.weight, mod.bias]
    opt = torch.optim.Adam(params, lr); Xt = T(X)
    for _ in range(CFG['tent_epochs']):
        m.train()
        for b in batches(len(Xt), 256):
            lp = F.log_softmax(m(Xt[b]), 1); loss = -(lp.exp() * lp).sum(1).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    return m.eval()

def shot(m, X, seed, lr):
    set_seed(seed); m = copy.deepcopy(m); m.h.requires_grad_(False)
    opt = torch.optim.Adam(m.f.parameters(), lr); Xt = T(X)
    for _ in range(CFG['shot_epochs']):
        m.eval()
        with torch.no_grad():
            feats = torch.cat([m.f(Xt[i:i + 8192]) for i in range(0, len(Xt), 8192)])
            p = F.softmax(m.h(feats), 1); fn = F.normalize(feats, dim=1)
            c = F.normalize((p.T @ fn) / (p.sum(0)[:, None] + 1e-8), dim=1); pl = (fn @ c.T).argmax(1)
            oh = F.one_hot(pl, 2).float(); c = F.normalize((oh.T @ fn) / (oh.sum(0)[:, None] + 1e-8), dim=1); pl = (fn @ c.T).argmax(1)
        m.train()
        for b in batches(len(Xt), 256):
            out = m(Xt[b]); q = F.softmax(out, 1); ent = -(q * torch.log(q + 1e-5)).sum(1).mean()
            qb = q.mean(0); div = (qb * torch.log(qb + 1e-5)).sum()
            loss = ent + div + CFG['shot_beta'] * F.cross_entropy(out, pl[b]); opt.zero_grad(); loss.backward(); opt.step()
    return m.eval()

def aad(m, X, seed):
    # Attracting-and-Dispersing (Yang et al., NeurIPS 2022): neighbours in feature space should agree, random pairs should not.
    set_seed(seed); m = copy.deepcopy(m); m.h.requires_grad_(False)
    opt = torch.optim.Adam(m.f.parameters(), CFG['aad_lr']); Xt = T(X); K = CFG['aad_k']
    m.eval()
    with torch.no_grad():
        feats = torch.cat([m.f(Xt[i:i + 8192]) for i in range(0, len(Xt), 8192)])
        FB = F.normalize(feats, dim=1); PB = F.softmax(m.h(feats), 1)
    total = CFG['aad_epochs'] * math.ceil(len(Xt) / 256); it = 0
    for _ in range(CFG['aad_epochs']):
        m.train()
        for b in batches(len(Xt), 256):
            feat = m.f(Xt[b]); q = F.softmax(m.h(feat), 1)
            with torch.no_grad():
                fn = F.normalize(feat, dim=1); FB[b] = fn; PB[b] = q.detach()
                sim = fn @ FB.T; sim[torch.arange(len(b), device=device), b] = -2
                nb = sim.topk(K, dim=1).indices; Pn = PB[nb]
            attract = -(q[:, None, :] * Pn).sum(-1).sum(1).mean()
            dot = q @ q.T; mask = 1 - torch.eye(len(b), device=device)
            disperse = (dot * mask).sum(1).mean() / (len(b) - 1) * K
            alpha = (1 + 10 * it / total) ** (-5.0); it += 1
            loss = attract + alpha * disperse; opt.zero_grad(); loss.backward(); opt.step()
    return m.eval()

def em_prior(P_unlab, P_eval, pi_s, iters=200):
    # Saerens et al. (2002): re-estimate the target class prior by EM from unlabeled target posteriors; needs only the source prior.
    pi_t = pi_s.copy()
    for _ in range(iters):
        q = P_unlab * (pi_t / pi_s); q /= q.sum(1, keepdims=True); new = q.mean(0)
        if np.abs(new - pi_t).max() < 1e-6: pi_t = new; break
        pi_t = new
    q = P_eval * (pi_t / pi_s); q /= q.sum(1, keepdims=True)
    return q.argmax(1), float(pi_t[1])

def metrics(y, pred):
    pos, neg = y == 1, y == 0
    tpr = ((pred == 1) & pos).sum() / max(pos.sum(), 1); fpr = ((pred == 1) & neg).sum() / max(neg.sum(), 1)
    return dict(macro_f1=f1_score(y, pred, average='macro', zero_division=0), attack_recall=float(tpr), fpr=float(fpr),
                true_attack_rate=float(y.mean()), pred_attack_rate=float(pred.mean()))

def resample(X, y, a, prior, n, seed):
    rs = np.random.RandomState(seed); att, ben = np.where(y == 1)[0], np.where(y == 0)[0]
    n = int(min(n, len(att) / prior, len(ben) / (1 - prior))); na = int(round(prior * n))
    idx = rs.permutation(np.concatenate([rs.choice(att, na, replace=False), rs.choice(ben, n - na, replace=False)]))
    return X[idx], y[idx], a[idx]

In [11]:
# ===== 4. Runner (resumable) =====
def run_all(data, feats, pairs, out_prefix, seeds, src_epochs, featsets):
    csv_m, csv_t = f'{out_prefix}_metrics.csv', f'{out_prefix}_per_type.csv'
    done = set()
    if os.path.exists(csv_m):
        d = pd.read_csv(csv_m); done = set(zip(d.featset, d.seed, d.src, d.tgt, d.prior.astype(str)))
    for fs in featsets:
        cols = [i for i, c in enumerate(feats) if fs == 'all' or c not in CFG['artifacts']]
        for seed in seeds:
            for s, t in pairs:
                todo = [p for p in CFG['priors'] if (fs, seed, s, t, str(p)) not in done]
                if not todo: continue
                S, Tg = data[s], data[t]
                mu, sd = S['Xtr'][:, cols].mean(0), S['Xtr'][:, cols].std(0) + 1e-6
                norm = lambda X: np.clip((X[:, cols] - mu) / sd, -10, 10).astype(np.float32)
                t0 = time.time(); src = train_sup(norm(S['Xtr']), S['ytr'], seed, src_epochs)
                in_dom = metrics(S['yte'], proba(src, norm(S['Xte'])).argmax(1))['macro_f1']
                pi_s = np.array([1 - S['ytr'].mean(), S['ytr'].mean()]); seen = set(S['atr'][S['ytr'] == 1])
                for prior in todo:
                    pe = CFG['attack_rate'][t] if prior == 'natural' else prior
                    Xa, ya_, _ = resample(Tg['Xtr'], Tg['ytr'], Tg['atr'], pe, CFG['n_target_train'], seed)
                    Xb, yb_, ab = resample(Tg['Xte'], Tg['yte'], Tg['ate'], pe, CFG['n_target_test'], seed + 100)
                    Xa, Xb = norm(Xa), norm(Xb)
                    preds, extra = {}, {}
                    preds['source_only'] = proba(src, Xb).argmax(1)
                    preds['bn_adapt'] = proba(bn_adapt(src, Xa), Xb).argmax(1)
                    for lr in CFG['tent_lrs']: preds[f'tent@{lr:g}'] = proba(tent(src, Xa, seed, lr), Xb).argmax(1)
                    for lr in CFG['shot_lrs']: preds[f'shot@{lr:g}'] = proba(shot(src, Xa, seed, lr), Xb).argmax(1)
                    preds['aad'] = proba(aad(src, Xa, seed), Xb).argmax(1)
                    preds['em_prior'], est = em_prior(proba(src, Xa), proba(src, Xb), pi_s); extra['em_prior'] = est
                    preds['oracle'] = proba(train_sup(Xa, ya_, seed, src_epochs), Xb).argmax(1)
                    rows, trows = [], []
                    for name, pred in preds.items():
                        rows.append(dict(featset=fs, seed=seed, src=s, tgt=t, prior=str(prior), method=name, src_in_domain_f1=in_dom,
                                         est_prior=extra.get(name, np.nan), **metrics(yb_, pred)))
                        for typ in sorted(set(ab[yb_ == 1])):
                            mk = ab == typ
                            trows.append(dict(featset=fs, seed=seed, src=s, tgt=t, prior=str(prior), method=name, attack_type=typ,
                                              seen_in_source=typ in seen, n=int(mk.sum()), recall=float((pred[mk] == 1).mean())))
                    pd.DataFrame(rows).to_csv(csv_m, mode='a', header=not os.path.exists(csv_m), index=False)
                    pd.DataFrame(trows).to_csv(csv_t, mode='a', header=not os.path.exists(csv_t), index=False)
                    r = {x['method']: round(x['macro_f1'], 3) for x in rows if '@' not in x['method'] or x['method'] in (f"tent@{CFG['default_tent_lr']:g}", f"shot@{CFG['default_shot_lr']:g}")}
                    print(f'[{fs}] seed {seed} {s}->{t} prior {prior}: {r} ({(time.time()-t0)/60:.1f} min)', flush=True)
    return pd.read_csv(csv_m), pd.read_csv(csv_t)

## Self-test on synthetic data (≈ 1 min) — must print SELF-TEST PASSED

In [12]:
# ===== 5. Self-test =====
def synth(n, shift, prior, seed):
    rs = np.random.RandomState(seed); y = (rs.rand(n) < prior).astype(np.int64)
    X = rs.randn(n, 8).astype(np.float32) + y[:, None] * 1.5 + shift
    return X, y, np.where(y == 1, np.where(rs.rand(n) < 0.5, 'dos', 'scan'), 'benign')
fake, ffeats = {}, ['MIN_TTL', 'A', 'B', 'C', 'D', 'E', 'F', 'G']
for i, (k, sh, pr) in enumerate([('A', 0.0, 0.6), ('B', 0.8, 0.1)]):
    X, y, a = synth(6000, sh, pr, i); fake[k] = dict(Xtr=X[2000:], ytr=y[2000:], atr=a[2000:], Xte=X[:2000], yte=y[:2000], ate=a[:2000])
_save = copy.deepcopy(CFG); CFG.update(n_target_train=2000, n_target_test=1000, attack_rate={'A': 0.6, 'B': 0.1}, aad_epochs=1, shot_epochs=1)
for f in glob.glob(f'{OUT}/selftest_*'): os.remove(f)
mt, tt = run_all(fake, ffeats, [('A', 'B')], f'{OUT}/selftest', [0], 2, ['all', 'no_artifacts'])
CFG.clear(); CFG.update(_save)
n_methods = 1 + 1 + 3 + 3 + 1 + 1 + 1
assert len(mt) == 2 * 2 * n_methods and len(tt) > 0, f'self-test incomplete: {len(mt)} rows'
print('SELF-TEST PASSED'); print(mt[['featset', 'prior', 'method', 'macro_f1', 'est_prior']].round(3).to_string(index=False))

[all] seed 0 A->B prior natural: {'source_only': 0.524, 'bn_adapt': 0.505, 'tent@0.001': 0.516, 'shot@0.0001': 0.518, 'aad': 0.517, 'em_prior': 0.953, 'oracle': 0.744} (0.0 min)
[all] seed 0 A->B prior 0.05: {'source_only': 0.454, 'bn_adapt': 0.398, 'tent@0.001': 0.424, 'shot@0.0001': 0.429, 'aad': 0.43, 'em_prior': 0.935, 'oracle': 0.669} (0.0 min)
[no_artifacts] seed 0 A->B prior natural: {'source_only': 0.528, 'bn_adapt': 0.512, 'tent@0.001': 0.517, 'shot@0.0001': 0.521, 'aad': 0.522, 'em_prior': 0.876, 'oracle': 0.641} (0.0 min)
[no_artifacts] seed 0 A->B prior 0.05: {'source_only': 0.454, 'bn_adapt': 0.402, 'tent@0.001': 0.423, 'shot@0.0001': 0.426, 'aad': 0.428, 'em_prior': 0.954, 'oracle': 0.535} (0.0 min)
SELF-TEST PASSED
     featset   prior      method  macro_f1  est_prior
         all natural source_only     0.524        NaN
         all natural    bn_adapt     0.505        NaN
         all natural tent@0.0001     0.516        NaN
         all natural  tent@0.001     0.516  

## Real data

In [13]:
# ===== 6. Load datasets (optional ones are skipped if their Kaggle slug is unavailable) =====
raw = {}
for k, slug in CFG['datasets'].items():
    try:
        raw[k] = load_nf(k, slug, CFG['n_per_dataset'])
    except Exception as e:
        if k in CFG['required']: raise
        print(f'SKIPPED optional dataset {k} ({slug}): {type(e).__name__}: {str(e)[:150]}')
DATA, FEATS = build_data(raw); del raw
json.dump(dict(features=FEATS, datasets=list(DATA)), open(f'{OUT}/datasets_used.json', 'w'), indent=2)

ton loaded from cache
unsw loaded from cache
bot loaded from cache
cic loaded from cache
ton: 300000 rows | types ['backdoor', 'ddos', 'dos', 'injection', 'mitm', 'password', 'ransomware', 'scanning', 'xss']
unsw: 270872 rows | types ['analysis', 'backdoor', 'dos', 'exploits', 'fuzzers', 'generic', 'reconnaissance', 'shellcode', 'worms']
bot: 300000 rows | types ['ddos', 'dos', 'reconnaissance', 'theft']
cic: 300000 rows | types ['bot', 'brute force -web', 'brute force -xss', 'ddos attack-hoic', 'ddos attack-loic-udp', 'ddos attacks-loic-http', 'dos attacks-goldeneye', 'dos attacks-hulk', 'dos attacks-slowhttptest', 'dos attacks-slowloris', 'ftp-bruteforce', 'infilteration', 'sql injection', 'ssh-bruteforce']
38 common features; artifacts present: ['MIN_TTL', 'MAX_TTL', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT']


In [14]:
# ===== 7. Run =====
keys = list(DATA); PAIRS = [(s, t) for s in keys for t in keys if s != t]
print(f'{len(PAIRS)} directions x {len(CFG["seeds"])} seeds x {len(CFG["priors"])} priors x {len(CFG["featsets"])} feature sets')
MT, TT = run_all(DATA, FEATS, PAIRS, f'{OUT}/real', CFG['seeds'], CFG['src_epochs'], CFG['featsets'])
print('done:', len(MT), 'metric rows')

12 directions x 3 seeds x 2 priors x 2 feature sets
[no_artifacts] seed 0 cic->ton prior 0.05: {'source_only': 0.457, 'bn_adapt': 0.395, 'tent@0.001': 0.406, 'shot@0.0001': 0.423, 'aad': 0.405, 'em_prior': 0.464, 'oracle': 0.891} (0.4 min)
[no_artifacts] seed 0 cic->unsw prior natural: {'source_only': 0.469, 'bn_adapt': 0.328, 'tent@0.001': 0.32, 'shot@0.0001': 0.387, 'aad': 0.34, 'em_prior': 0.509, 'oracle': 0.965} (0.4 min)
[no_artifacts] seed 0 cic->unsw prior 0.05: {'source_only': 0.471, 'bn_adapt': 0.335, 'tent@0.001': 0.331, 'shot@0.0001': 0.388, 'aad': 0.372, 'em_prior': 0.511, 'oracle': 0.971} (0.7 min)
[no_artifacts] seed 0 cic->bot prior natural: {'source_only': 0.307, 'bn_adapt': 0.378, 'tent@0.001': 0.382, 'shot@0.0001': 0.305, 'aad': 0.378, 'em_prior': 0.067, 'oracle': 0.908} (0.4 min)
[no_artifacts] seed 0 cic->bot prior 0.05: {'source_only': 0.273, 'bn_adapt': 0.346, 'tent@0.001': 0.35, 'shot@0.0001': 0.278, 'aad': 0.384, 'em_prior': 0.142, 'oracle': 0.988} (0.7 min)
[no

In [15]:
# ===== 8. Analysis =====
MT = pd.read_csv(f'{OUT}/real_metrics.csv'); TT = pd.read_csv(f'{OUT}/real_per_type.csv')
MT['prior'] = MT.prior.astype(str); TT['prior'] = TT.prior.astype(str)
dt, ds = f"tent@{CFG['default_tent_lr']:g}", f"shot@{CFG['default_shot_lr']:g}"
key = ['featset', 'seed', 'src', 'tgt', 'prior']
W = MT.pivot_table(index=key, columns='method', values='macro_f1')
W['tent_tuned_best'] = W[[c for c in W if c.startswith('tent@')]].max(1)    # chosen WITH target labels: unreachable upper bound
W['shot_tuned_best'] = W[[c for c in W if c.startswith('shot@')]].max(1)
METHODS = ['source_only', 'bn_adapt', dt, ds, 'aad', 'em_prior', 'tent_tuned_best', 'shot_tuned_best', 'oracle']
G = W[METHODS].groupby(['featset', 'src', 'tgt', 'prior']).mean()
G['gap'] = G['oracle'] - G['source_only']
print('Mean macro-F1 over seeds:'); print(G.round(3).to_string())
G.to_csv(f'{OUT}/macro_f1_by_direction.csv')

R = G.copy()
for m in METHODS[1:-1]: R[m] = (G[m] - G['source_only']) / G['gap'].where(G['gap'] > 0.05)
R = R[METHODS[1:-1]]
print('\nFraction of the gap recovered (1 = reaches oracle, <0 = worse than no adaptation):'); print(R.round(2).to_string())

S = []
for (fs, pr), g in R.groupby(level=['featset', 'prior']):
    for m in METHODS[1:-1]:
        v = g[m].dropna(); h = (G.loc[g.index, m] < G.loc[g.index, 'source_only'] - CFG['harm_margin']).sum()
        S.append(dict(featset=fs, prior=pr, method=m, directions=len(v), median_recovery=round(float(v.median()), 2) if len(v) else np.nan,
                      n_recovery_below_half=int((v < CFG['recovery_min']).sum()), n_harmed=int(h)))
S = pd.DataFrame(S); S.to_csv(f'{OUT}/robustness_summary.csv', index=False)
print('\nRobustness summary:'); print(S.to_string(index=False))

C = MT[MT.method.isin(['source_only', 'bn_adapt', dt, ds, 'aad', 'em_prior'])].copy()
C['collapse_benign'] = (C.pred_attack_rate < 0.01) & (C.true_attack_rate > 0.03)
C['false_alarm_flood'] = C.fpr > 0.2
print('\nShare of runs with collapse to all-benign / false-alarm rate > 20%:')
print(C.groupby(['featset', 'method'])[['collapse_benign', 'false_alarm_flood']].mean().round(2).to_string())

E = MT[MT.method == 'em_prior'].groupby(['featset', 'src', 'tgt', 'prior'])[['est_prior', 'true_attack_rate']].mean().round(3)
print('\nEM label-shift: estimated vs true target attack rate:'); print(E.to_string())

U = TT[TT.method.isin(['source_only', dt, ds, 'aad'])].groupby(['featset', 'method', 'seen_in_source']).recall.mean().unstack().round(3)
print('\nAttack-type recall, seen vs unseen by the source:'); print(U.to_string())

A = G.xs('natural', level='prior')['source_only'].unstack('featset') if 'no_artifacts' in G.index.get_level_values(0) else None
if A is not None:
    print('\nArtifact ablation — source-only macro-F1 (natural prior), all vs no_artifacts:'); print(A.round(3).to_string())
json.dump(S.to_dict('records'), open(f'{OUT}/robustness_summary.json', 'w'), indent=1)

Mean macro-F1 over seeds:
method                          source_only  bn_adapt  tent@0.001  shot@0.0001    aad  em_prior  tent_tuned_best  shot_tuned_best  oracle    gap
featset      src  tgt  prior                                                                                                                    
all          bot  cic  0.05           0.545     0.288       0.307        0.461  0.327     0.547            0.314            0.487   0.984  0.439
                       natural        0.569     0.362       0.380        0.568  0.395     0.547            0.390            0.595   0.985  0.416
                  ton  0.05           0.278     0.318       0.321        0.284  0.334     0.274            0.323            0.308   0.927  0.649
                       natural        0.440     0.496       0.520        0.379  0.436     0.438            0.528            0.399   0.965  0.525
                  unsw 0.05           0.542     0.299       0.280        0.458  0.342     0.539         